# Quantum Delivery Route Planner
## QAOA-Based TSP Solver — Simulation Analysis Notebook

This notebook walks through the **complete end-to-end quantum optimization pipeline**
for solving the **Traveling Salesperson Problem (TSP)** using the
**Quantum Approximate Optimization Algorithm (QAOA)** on a Qiskit Aer simulator.

---

### Pipeline Overview

| Step | Module | Purpose |
|------|--------|---------|
| 1 | `problem_encoder.py` | Build distance matrix |
| 2 | `qubo_builder.py` | Build TSP QUBO formulation |
| 3 | `ising_converter.py` | Convert to Ising Hamiltonian |
| 4 | `qaoa.py` | Run QAOA optimization |
| 5 | `simulator.py` | Execute QAOA circuit on Aer |
| 6 | `decoder.py` | Decode bitstring to route |
| 7 | `result_processor.py` | Compute statistics |
| 8 | `tsp_solver.py` | Classical OR-Tools benchmark |
| 9 | `compare_results.py` | Compare results |
| 10 | Visualization | Charts and plots |

## 1. Setup & Imports

In [ ]:
import json
import pickle
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

# Set project root
BASE_DIR = Path('.').resolve().parent
sys.path.insert(0, str(BASE_DIR))

print('Project Root:', BASE_DIR)

# Matplotlib dark style
plt.rcParams.update({
    'figure.facecolor': '#1a1a2e',
    'axes.facecolor': '#16213e',
    'axes.labelcolor': 'white',
    'xtick.color': 'white',
    'ytick.color': 'white',
    'text.color': 'white',
    'axes.edgecolor': '#4a4a8a',
    'grid.color': '#2a2a5a',
    'grid.alpha': 0.4,
})
print('Setup complete.')

## 2. Data Loading & Distance Matrix

In [ ]:
# Load delivery locations
locations_file = BASE_DIR / 'data' / 'raw' / 'delivery_locations.csv'
locations = pd.read_csv(locations_file)

print('=' * 60)
print('Delivery Locations')
print('=' * 60)
display(locations)

# Load distance matrix
dist_file = BASE_DIR / 'data' / 'processed' / 'distance_matrix.csv'
dist_matrix = pd.read_csv(dist_file, index_col=0)

print('\nDistance Matrix (Euclidean):')
display(dist_matrix.round(2))

In [ ]:
import networkx as nx

G = nx.Graph()
positions = {}

for _, row in locations.iterrows():
    node = int(row['location_id'])
    G.add_node(node, name=row['name'], ntype=row['type'])
    positions[node] = (row['x'], row['y'])

nodes = list(G.nodes())
for i in range(len(nodes)):
    for j in range(i+1, len(nodes)):
        G.add_edge(nodes[i], nodes[j])

fig, ax = plt.subplots(figsize=(9, 7))

depot_nodes = [n for n, d in G.nodes(data=True) if d['ntype'] == 'Depot']
customer_nodes = [n for n, d in G.nodes(data=True) if d['ntype'] == 'Customer']
labels = {n: d['name'] for n, d in G.nodes(data=True)}

nx.draw_networkx_edges(G, positions, alpha=0.2, edge_color='#4a4a8a')
nx.draw_networkx_nodes(G, positions, nodelist=customer_nodes,
                       node_size=500, node_color='#2196F3', label='Customer')
nx.draw_networkx_nodes(G, positions, nodelist=depot_nodes,
                       node_size=800, node_color='#FF5722', node_shape='s', label='Depot')
nx.draw_networkx_labels(G, positions, labels, font_color='white', font_size=9)

ax.set_title('Delivery Network — Complete Graph', fontsize=14, pad=15)
ax.legend(facecolor='#1a1a2e', labelcolor='white')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. QUBO & Ising Formulation

In [ ]:
from optimization.qubo_builder import TSPQUBOBuilder
from qiskit_optimization.converters import QuadraticProgramToQubo

dataset = BASE_DIR / 'data' / 'processed' / 'validated_vrp_dataset.json'

builder = TSPQUBOBuilder(dataset)
qp = builder.build()

print('=' * 60)
print('Quadratic Program (TSP)')
print('=' * 60)
print(qp.prettyprint())

In [ ]:
# Convert to QUBO
converter = QuadraticProgramToQubo()
qubo = converter.convert(qp)

print(f'QUBO Variables  : {qubo.get_num_binary_vars()}')
print(f'QUBO Constraints: {qubo.get_num_linear_constraints()}')

# Convert to Ising
operator, offset = qubo.to_ising()

print()
print('=' * 60)
print('Ising Hamiltonian')
print('=' * 60)
print(f'Number of Qubits : {operator.num_qubits}')
print(f'Offset           : {offset:.4f}')
print()
print('Pauli Operator (first 5 terms):')
for i, (coeff, pauli) in enumerate(zip(operator.coeffs, operator.paulis)):
    if i >= 5:
        break
    print(f'  {coeff:.4f} * {pauli}')

## 4. QAOA Optimization Results

In [ ]:
# Load QAOA result
qaoa_file = BASE_DIR / 'results' / 'simulator' / 'qaoa_result.pkl'

with open(qaoa_file, 'rb') as f:
    qaoa_result = pickle.load(f)

print('=' * 60)
print('QAOA Optimization Result')
print('=' * 60)
print(f"Optimizer      : {qaoa_result['optimizer']}")
print(f"QAOA Reps      : {qaoa_result['reps']}")
print(f"Num Qubits     : {qaoa_result['num_qubits']}")
print(f"Eigenvalue     : {qaoa_result['eigenvalue']:.6f}")
print(f"Optimal Point  : {qaoa_result['optimal_point']}")
print(f"Max Iterations : {qaoa_result['maxiter']}")

## 5. Quantum Simulation & Measurement Results

In [ ]:
# Load simulation counts
counts_file = BASE_DIR / 'results' / 'simulator' / 'counts.json'

with open(counts_file, 'r') as f:
    counts = json.load(f)

total_shots = sum(counts.values())
probs = {k: v / total_shots for k, v in counts.items()}

print(f'Total Shots      : {total_shots:,}')
print(f'Unique Bitstrings: {len(counts):,}')

top_10 = sorted(counts.items(), key=lambda x: x[1], reverse=True)[:10]
print('\nTop 10 Measurement Outcomes:')
print(f'{"Bitstring":<22} {"Count":<10} {"Probability"}')
print('-' * 45)
for bs, cnt in top_10:
    print(f'{bs[:22]:<22} {cnt:<10} {cnt/total_shots:.6f}')

In [ ]:
# Plot probability distribution
top_n = 20
top = sorted(probs.items(), key=lambda x: x[1], reverse=True)[:top_n]
labels_bs = [x[0][:8] + '..' if len(x[0]) > 8 else x[0] for x in top]
values_pr = [x[1] for x in top]

fig, ax = plt.subplots(figsize=(14, 5))
colors = ['#FF5722' if i == 0 else '#9C27B0' for i in range(len(labels_bs))]
ax.bar(range(len(labels_bs)), values_pr, color=colors, edgecolor='#333')
ax.set_xticks(range(len(labels_bs)))
ax.set_xticklabels(labels_bs, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Probability')
ax.set_title(f'QAOA Measurement Probability Distribution (Top {top_n})', fontsize=13)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Classical TSP Benchmark (OR-Tools)

In [ ]:
classical_file = BASE_DIR / 'results' / 'classical' / 'classical_result.json'

with open(classical_file, 'r') as f:
    classical = json.load(f)

print('=' * 60)
print('Classical TSP Result')
print('=' * 60)
print(f"Solver      : {classical['solver']}")
print(f"Route Cost  : {classical['route_cost']:.4f}")
print(f"Exec Time   : {classical['execution_time']:.6f}s")
print(f"\nRoute: {' -> '.join(classical['route_names'])}")

## 7. Classical vs Quantum Comparison

In [ ]:
comp_file = BASE_DIR / 'results' / 'comparison' / 'comparison_result.json'

with open(comp_file, 'r') as f:
    comp = json.load(f)

perf = comp['performance']
cost_m = perf['cost_comparison']
speed_m = perf['speed_comparison']
qual_m = perf['solution_quality']
div_m = perf['solution_diversity']

table = pd.DataFrame({
    'Metric': ['Route Cost', 'Exec Time (s)', 'Unique Solutions', 'Accuracy', 'Optimality Gap'],
    'Classical (OR-Tools)': [
        f"{cost_m['classical_cost']:.4f}",
        f"{speed_m['classical_time_s']:.6f}",
        '1',
        '100% (reference)',
        '0%'
    ],
    'Quantum (QAOA)': [
        f"{cost_m['quantum_cost']:.4f}",
        f"{speed_m['quantum_time_s']:.4f}",
        str(comp['quantum']['unique_solutions']),
        f"{qual_m['accuracy_pct']:.2f}%",
        f"{qual_m['optimality_gap_pct']:.2f}%"
    ]
})
display(table)

In [ ]:
q_cost_val = comp['quantum']['route_cost']
c_cost_val = comp['classical']['route_cost']
q_time_val = comp['quantum']['execution_time']
c_time_val = comp['classical']['execution_time']
q_uniq_val = comp['quantum']['unique_solutions']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Classical vs Quantum QAOA Comparison', fontsize=14, fontweight='bold')

labels = ['Classical', 'QAOA']
colors_bar = ['#2196F3', '#FF5722']

# Cost
ax = axes[0]
bars = ax.bar(labels, [c_cost_val, q_cost_val], color=colors_bar, edgecolor='white', linewidth=0.5)
for b, v in zip(bars, [c_cost_val, q_cost_val]):
    ax.text(b.get_x() + b.get_width()/2, b.get_height(), f'{v:.2f}',
            ha='center', va='bottom', fontweight='bold')
ax.set_title('Route Cost', fontsize=12)
ax.set_ylabel('Total Distance')
ax.grid(axis='y', alpha=0.3)

# Time
ax = axes[1]
bars = ax.bar(labels, [c_time_val, q_time_val], color=colors_bar, edgecolor='white', linewidth=0.5)
for b, v in zip(bars, [c_time_val, q_time_val]):
    ax.text(b.get_x() + b.get_width()/2, b.get_height(), f'{v:.3f}s',
            ha='center', va='bottom', fontweight='bold')
ax.set_title('Execution Time', fontsize=12)
ax.set_ylabel('Seconds')
ax.grid(axis='y', alpha=0.3)

# Diversity
ax = axes[2]
bars = ax.bar(labels, [1, q_uniq_val], color=colors_bar, edgecolor='white', linewidth=0.5)
for b, v in zip(bars, [1, q_uniq_val]):
    ax.text(b.get_x() + b.get_width()/2, b.get_height(), f'{v:,}',
            ha='center', va='bottom', fontweight='bold')
ax.set_title('Unique Solutions Found', fontsize=12)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Conclusions & Research Summary

### Key Findings

| Aspect | Observation |
|--------|-------------|
| **QAOA Solution Quality** | QAOA finds an approximate solution with a small optimality gap on small instances (4 nodes) |
| **Classical Speed** | OR-Tools/DP is orders of magnitude faster due to no quantum simulation overhead |
| **Solution Diversity** | QAOA explores the entire solution space probabilistically, yielding many candidate routes |
| **Scalability Potential** | Quantum advantage expected at larger scales (50+ nodes) where classical algorithms become NP-hard |
| **Circuit Depth** | QAOA with `reps=2` produces moderate depth circuits suitable for near-term hardware |

### Research Value

This simulation demonstrates:
- **End-to-end quantum optimization pipeline** on real delivery data
- **QUBO/Ising formulation** of the TSP problem
- **QAOA variational algorithm** with COBYLA classical optimizer
- **Fair comparison** with exact classical solver
- **IBM Quantum readiness** — the same circuit can be submitted to real hardware

---
*Quantum Delivery Route Planner Capstone — Simulation Analysis*